# Notebook 04 — Churn Risk Segmentation

## What this notebook does
Converts the 1-5 RFM scores from Notebook 03 into six named
business segments. Produces a segment summary table that directly
answers the north star question:
"Which customers are most likely to never buy again —
and how much revenue walks out the door if we do nothing?"

## The six segments
Champions  : r>=4 AND f>=4  — recent, frequent, high value. Best customers.
At-Risk    : r<=2 AND f>=3  — used to buy often, now gone quiet. Primary target.
Lost       : r=1  AND f<=2  — long gone, low history. Hard to recover.
Promising  : r>=4, not champion — recent but infrequent. Growing relationship.
New        : f=1            — only one order ever. Too early to judge.
Loyal      : everything else — reliable buyers, not top-tier recency.

## Source → Target
Source : brazilian.gold.customer_rfm_scores (managed Delta)
Target : brazilian.gold.churn_segments      (managed Delta)
         brazilian.gold.segment_summary     (managed Delta)

## Why two output tables
churn_segments  : one row per customer with segment label — used for the retention list
segment_summary : one row per segment with aggregated KPIs — used for charts and exec summary

In [0]:
# ── SETUP ─────────────────────────────────────────────────────
from pyspark.sql import functions as F

spark.sql("USE CATALOG brazilian")

SOURCE_TABLE    = "brazilian.gold.customer_rfm_scores"
TARGET_SEGMENTS = "brazilian.gold.churn_segments"
TARGET_SUMMARY  = "brazilian.gold.segment_summary"

print(f"Reading from : {SOURCE_TABLE}")
print(f"Writing to   : {TARGET_SEGMENTS}")
print(f"             : {TARGET_SUMMARY}")

In [0]:
# ── UNDERSTAND THE SCORE DISTRIBUTION FIRST ───────────────────
#
# WHY THIS CELL BEFORE SEGMENTING:
# Before applying CASE WHEN rules, we need to understand how
# customers are distributed across RFM scores.
# This tells us whether our segment thresholds will produce
# meaningful group sizes or collapse everything into one bucket.
#
# A good segment distribution looks like:
# At-Risk    : 15-30% of customers  — large enough to act on
# Champions  : 5-15%  of customers  — small elite group
# Lost       : 15-30% of customers  — significant but lower priority
# No segment should be 0% or 80%+ — that signals threshold issues

print("Score distribution before segmenting:")
print("This tells us what our segment sizes will look like.\n")

spark.sql(f"""
    SELECT
        r_score,
        f_score,
        COUNT(*) AS customers
    FROM {SOURCE_TABLE}
    GROUP BY r_score, f_score
    ORDER BY r_score DESC, f_score DESC
""").show(25)

# Quick check of extreme score combinations
print("Customers with r_score=1 (least recent):")
r1 = spark.sql(f"""
    SELECT COUNT(*) AS count
    FROM {SOURCE_TABLE}
    WHERE r_score = 1
""").collect()[0][0]
print(f"  {r1:,} customers have not purchased recently\n")

print("Customers with r>=4 AND f>=4 (potential Champions):")
champions_check = spark.sql(f"""
    SELECT COUNT(*) AS count
    FROM {SOURCE_TABLE}
    WHERE r_score >= 4 AND f_score >= 4
""").collect()[0][0]
print(f"  {champions_check:,} customers qualify as Champions\n")

print("Customers with r<=2 AND f>=3 (potential At-Risk):")
at_risk_check = spark.sql(f"""
    SELECT COUNT(*) AS count
    FROM {SOURCE_TABLE}
    WHERE r_score <= 2 AND f_score >= 3
""").collect()[0][0]
print(f"  {at_risk_check:,} customers qualify as At-Risk")

In [0]:
# ── SEGMENT ASSIGNMENT ────────────────────────────────────────
#
# HOW THE CASE WHEN LOGIC WORKS:
# Spark evaluates conditions TOP TO BOTTOM and assigns the FIRST
# condition that is true. Order matters.
#
# We put Champions first because they satisfy multiple conditions
# (high r AND high f) — we want them captured before the others.
#
# SEGMENT DEFINITIONS AND BUSINESS REASONING:
#
# CHAMPIONS (r>=4, f>=4):
#   Most recent buyers who also buy frequently.
#   These are your best customers — protect them.
#   Action: loyalty rewards, early access, referral programs.
#
# AT-RISK (r<=2, f>=3):
#   They used to buy often but have gone quiet recently.
#   This is your PRIMARY intervention target — they know your brand,
#   they liked it enough to return multiple times, but something changed.
#   The revenue-at-risk number comes from this segment.
#   Action: personal outreach, win-back offers, satisfaction surveys.
#
# LOST (r=1, f<=2):
#   Bought once or twice a long time ago and never came back.
#   Hard to recover — low frequency means weak brand connection.
#   Low ROI to pursue aggressively, but worth a generic re-engagement.
#   Action: low-cost automated email campaign.
#
# PROMISING (r>=4, not champion):
#   Recently purchased but not yet frequent.
#   Could become Champions with nurturing.
#   Action: second-purchase incentive, product recommendations.
#
# NEW (f=1):
#   Only one order ever placed.
#   Too early to judge churn vs loyalty.
#   Action: onboarding sequence, encourage second purchase.
#
# LOYAL (everything else):
#   Regular buyers who are not top-tier on recency.
#   Solid base — maintain engagement.
#   Action: standard retention emails, loyalty program.

rfm = spark.table(SOURCE_TABLE)

segmented = rfm.withColumn("segment",

    # CHAMPIONS: recent AND frequent
    # Both conditions must be true — these customers are active and valuable
    F.when(
        (F.col("r_score") >= 4) & (F.col("f_score") >= 4),
        "Champions"

    # AT-RISK: not recent BUT historically frequent
    # The gap between their purchase history and their silence
    # is the signal — they know the brand but stopped engaging
    ).when(
        (F.col("r_score") <= 2) & (F.col("f_score") >= 3),
        "At-Risk"

    # LOST: very low recency AND low frequency
    # They barely engaged and have been silent the longest
    ).when(
        (F.col("r_score") == 1) & (F.col("f_score") <= 2),
        "Lost"

    # PROMISING: recent but not frequent enough for Champion
    # The relationship is new and growing
    ).when(
        F.col("r_score") >= 4,
        "Promising"

    # NEW: only one order ever placed
    # frequency=1 regardless of when it happened
    ).when(
        F.col("f_score") == 1,
        "New"

    # LOYAL: everyone else
    # Reliable but not elite — middle ground
    ).otherwise("Loyal")
)

# Preview the assignment before writing
print("Segment assignment preview (first 10 rows):")
segmented.select(
    "customer_unique_id",
    "r_score", "f_score", "m_score",
    "rfm_total", "monetary_value",
    "recency_days", "segment"
).show(10, truncate=False)

In [0]:
# ── WRITE churn_segments TABLE ────────────────────────────────
#
# This table has one row per customer with their segment label.
# Notebook 05 reads this table to build the retention priority list.
# Every customer in At-Risk and Lost feeds into the revenue-at-risk number.

(segmented
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TARGET_SEGMENTS))

written_count = spark.table(TARGET_SEGMENTS).count()
print(f"✅ {TARGET_SEGMENTS}")
print(f"   Rows written : {written_count:,} customers with segment labels")

In [0]:
# ── BUILD AND WRITE segment_summary TABLE ─────────────────────
#
# WHY A SEPARATE SUMMARY TABLE:
# churn_segments has one row per customer — 90K+ rows.
# For charts, dashboards, and the executive summary we need
# aggregated KPIs per segment — 6 rows.
# Keeping them separate means charts query the 6-row summary
# instead of scanning 90K rows every time.
#
# WHAT EACH COLUMN TELLS US:
# customers         → segment size — how many people
# pct_of_total      → relative size — what % of customer base
# avg_ltv           → average lifetime value per customer in segment
# total_spend       → total historical revenue from this segment
# avg_recency_days  → how long ago they last purchased on average
# avg_review_score  → satisfaction level of this segment
# avg_orders        → how many times they buy on average

segment_summary = spark.sql(f"""
    SELECT
        segment,

        -- SIZE
        COUNT(*)                                                      AS customers,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1)           AS pct_of_total,

        -- VALUE
        ROUND(AVG(monetary_value), 2)                                AS avg_ltv,
        ROUND(SUM(monetary_value), 2)                                AS total_spend,

        -- BEHAVIOR
        ROUND(AVG(recency_days), 0)                                  AS avg_recency_days,
        ROUND(AVG(frequency), 2)                                     AS avg_orders,

        -- SATISFACTION
        ROUND(AVG(avg_review_score), 2)                              AS avg_review_score,

        -- SCORE AVERAGES (for understanding what drove the segment)
        ROUND(AVG(r_score), 2)                                       AS avg_r_score,
        ROUND(AVG(f_score), 2)                                       AS avg_f_score,
        ROUND(AVG(m_score), 2)                                       AS avg_m_score

    FROM {TARGET_SEGMENTS}
    GROUP BY segment
    ORDER BY total_spend DESC
""")

(segment_summary
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TARGET_SUMMARY))

print(f"✅ {TARGET_SUMMARY}")
print(f"   Rows written : {segment_summary.count()} segments")
print()
print("SEGMENT DISTRIBUTION:")
segment_summary.show(truncate=False)

In [0]:
# ── VALIDATE SEGMENT DISTRIBUTION ────────────────────────────
#
# WHY VALIDATE:
# If a segment has 0 customers or 95% of all customers,
# the thresholds are wrong and need adjustment.
# These expected ranges come from RFM literature and
# are consistent with real e-commerce datasets.
#
# If your numbers fall outside these ranges do NOT adjust
# the ranges to match — instead revisit the CASE WHEN thresholds
# in Cell 4 and understand why the distribution looks different.

print("SEGMENT VALIDATION — checking distribution against expected ranges")
print()

spark.sql(f"""
    SELECT
        segment,
        customers,
        pct_of_total,
        CASE
            WHEN segment = 'Champions' AND pct_of_total BETWEEN 3  AND 20
                THEN '✅ Normal range'
            WHEN segment = 'At-Risk'   AND pct_of_total BETWEEN 10 AND 40
                THEN '✅ Normal range'
            WHEN segment = 'Lost'      AND pct_of_total BETWEEN 10 AND 40
                THEN '✅ Normal range'
            WHEN segment = 'New'       AND pct_of_total BETWEEN 10 AND 35
                THEN '✅ Normal range'
            WHEN segment = 'Loyal'     AND pct_of_total BETWEEN 5  AND 30
                THEN '✅ Normal range'
            WHEN segment = 'Promising' AND pct_of_total BETWEEN 3  AND 25
                THEN '✅ Normal range'
            ELSE '⚠  Review thresholds in Cell 4'
        END AS distribution_check,
        avg_ltv,
        avg_recency_days,
        avg_review_score
    FROM {TARGET_SUMMARY}
    ORDER BY customers DESC
""").show(truncate=False)

In [0]:
# ── THE KEY FINDING — SATISFACTION VS CHURN ───────────────────
#
# WHY THIS ANALYSIS:
# Standard RFM only uses purchase behavior.
# We added review_score as a fourth dimension.
# If At-Risk and Lost customers have lower review scores than
# Champions, that proves satisfaction drives churn — not just
# purchase frequency.
# This is an analytical insight that goes BEYOND the job description
# and signals mature thinking to any interviewer.
#
# HOW TO USE THIS IN INTERVIEWS:
# "Beyond standard RFM scoring, I cross-referenced churn segments
#  with customer review scores and found that At-Risk customers
#  average X stars versus Champions at Y stars. This suggests
#  satisfaction issues are a leading indicator of churn, not just
#  a lagging one. A business implication is that the retention
#  campaign should address service quality, not just offer discounts."

print("KEY FINDING — Satisfaction score across churn segments")
print("If At-Risk/Lost score lower, satisfaction drives churn.\n")

spark.sql(f"""
    SELECT
        segment,
        avg_review_score,
        customers,
        avg_recency_days,
        avg_ltv,
        ROUND(avg_review_score - AVG(avg_review_score) OVER(), 2) AS vs_overall_avg
    FROM {TARGET_SUMMARY}
    ORDER BY avg_review_score DESC
""").show(truncate=False)

print()
print("INTERPRETATION:")
print("  Segments with negative vs_overall_avg have below-average satisfaction.")
print("  If At-Risk and Lost are negative → satisfaction drives churn.")
print("  Use this finding in your README and LinkedIn post.")

In [0]:
# ── FINAL SUMMARY ─────────────────────────────────────────────

total_customers = spark.sql(f"""
    SELECT COUNT(*) AS n FROM {TARGET_SEGMENTS}
""").collect()[0][0]

at_risk_n = spark.sql(f"""
    SELECT COUNT(*) AS n FROM {TARGET_SEGMENTS}
    WHERE segment = 'At-Risk'
""").collect()[0][0]

lost_n = spark.sql(f"""
    SELECT COUNT(*) AS n FROM {TARGET_SEGMENTS}
    WHERE segment = 'Lost'
""").collect()[0][0]

revenue_at_risk = spark.sql(f"""
    SELECT ROUND(SUM(monetary_value), 2) AS r
    FROM {TARGET_SEGMENTS}
    WHERE segment IN ('At-Risk', 'Lost')
""").collect()[0][0]

print("=" * 65)
print("SEGMENTATION COMPLETE")
print("=" * 65)
print(f"  Total customers segmented : {total_customers:,}")
print(f"  At-Risk customers         : {at_risk_n:,}")
print(f"  Lost customers            : {lost_n:,}")
print(f"  Combined at-risk          : {at_risk_n + lost_n:,}")
print(f"  Revenue at risk           : ${revenue_at_risk:,.2f}")
print("=" * 65)
print()
print("  These are the numbers you lead every interview with.")
print("  Write them down. Memorize them.")
print()
print("  Tables written:")
print(f"  {TARGET_SEGMENTS}  — one row per customer")
print(f"  {TARGET_SUMMARY}      — one row per segment (6 rows)")
print()
print("  Next: 05_revenue_at_risk.py")
print("=" * 65)